# サスペンション速度分析

このノートブックでは、ショックポット変位データからサスペンション速度分布を分析します。ダンパーが異なる速度域でどのように動作しているかを理解することは、適切なセットアップに不可欠です。

## 解釈ガイド

| パターン | 考えられる原因 | 対処法 |
|---------|----------------|--------|
| 高い摩擦率 | ダンパー摩擦、低速減衰不足 | ダンパーシールを確認、低速側を柔らかく |
| 正の歪度（バンプ側が多い） | ノーズダイブ、ロール剛性バランスの偏り | 圧側減衰を強化、スプリングレートを確認 |
| 負の歪度（リバウンド側が多い） | リアスクワット、伸び側の問題 | 伸側減衰を強化、スプリングプリロードを確認 |
| 高い縁石率 | 縁石の積極的使用、底付き | バンプストップ追加、スプリングレートを上げる |
| 左右の非対称性 | 重量配分、アライメント | コーナーウェイト、アライメント、ダンパー設定を確認 |
| 前後の非対称性 | ピッチバランスの問題 | 前後の減衰またはスプリング比を調整 |

## このノートブックの内容

- **速度ヒストグラム**: 各コーナー（FL、FR、RL、RR）のホイール速度分布
- **速度域シェーディング**: ダンパーの動作域を視覚的に表示
  - **グレー（摩擦）**: < 5 mm/s - 静止摩擦領域、ダンパーが動かない可能性あり
  - **水色（低速）**: 5-25 mm/s - 低速減衰、車体コントロールに影響
  - **黄緑（高速）**: 25-200 mm/s - 高速減衰、バンプ吸収に影響
  - **薄い赤（縁石）**: > 200 mm/s - 縁石衝撃、底付き
- **統計テーブル**: 歪度、対称性、コーナー比較

## 自分のデータを使用する場合

1. 下の**最初のセルを実行**してパッケージをインストールし、アップロードウィジェットを表示
2. **「Choose File」をクリック**して`.xrk`、`.xrz`、または`.ibt`ファイルを選択
3. ショックポットチャンネルの名前が異なる場合は**チャンネル名を設定**
4. **残りのセルをすべて実行**してデータを分析

## 必要なチャンネル

- ショックポット変位チャンネル（例：`LF_Shock_Pot`、`RF_Shock_Pot`など）

**注意:** このノートブックはJupyterLite（ブラウザ）と通常のJupyterLab環境の両方で動作します。

In [1]:
# 必要なパッケージをインストール（JupyterLiteで必要、通常のJupyterLabでは既にインストール済みならスキップ）
%pip install -q pandas plotly libxrk libibt motorsports-data-notebook jinja2 ipywidgets

# Rustパーサーバックエンドを使用（ファイル読み込みが約3倍高速）
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# ヘルパー関数をインポート
from motorsports_data_notebook.suspension import (
    MotionRatios,
    VelocityRanges,
    SUSPENSION_CHANNEL_NAMES,
    analyze_suspension_velocity,
    format_suspension_stats_table,
    format_symmetry_table,
    format_comparison_table,
)
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_suspension_velocity_histogram,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker

# セッションピッカーとチャンネル設定
# 自分のファイルをアップロードして分析するラップを選択
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        "shock_fl": "LF_Shock_Pot",
        "shock_fr": "RF_Shock_Pot",
        "shock_rl": "LR_Shock_Pot",
        "shock_rr": "RR_Shock_Pot",
    },
)
session.display()

/home/runner/work/motorsports_data_notebook/motorsports_data_notebook/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ラップ情報をpandas DataFrameとして取得
laps = session.get_laps()

In [3]:
# ラップタイム一覧を表示
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

,num,start_time,end_time,lap_time
0,1,150454,279602,2:09.148
1,2,279602,406240,2:06.638
2,3,406240,532797,2:06.557
3,4,532797,659283,2:06.486
4,5,659283,787773,2:08.490
5,6,787773,913776,2:06.003
6,7,913776,1041398,2:07.622
7,8,1041398,1168323,2:06.925
8,9,1168323,1294676,2:06.353
9,10,1294676,1420573,2:05.897


## 設定

以下でショックポットのチャンネル名とモーションレシオを設定します。デフォルトはToyota 86 ZN6と標準的なAIMチャンネル名に設定されています。

In [4]:
# 設定されたチャンネル名を取得
channel_names = session.get_channel_names()

# モーションレシオ: ホイール速度 = ショック速度 / モーションレシオ
# デフォルト: Toyota 86 ZN6（フロントマクファーソン: 0.997、リアマルチリンク: 0.768）
motion_ratios = MotionRatios(
    front_left=0.997,
    front_right=0.997,
    rear_left=0.768,
    rear_right=0.768,
)

# 速度域しきい値（mm/s）- シェーディング領域を定義
velocity_ranges = VelocityRanges(
    friction=5.0,  # 以下: 静止摩擦領域
    slow=25.0,  # 以下: 低速減衰
    fast=200.0,  # 以上: 縁石/高速
)

## ベストラップの分析

In [5]:
# 選択したラップを取得
selected_lap = session.get_selected_lap()
log = session.get_log()
lap_num = int(selected_lap["num"])
print(f"ラップ {lap_num} を分析中 - {format_lap_time(selected_lap['lap_time'])}")

# libxrk 0.5.0のメソッドを使用して選択したラップでログをフィルタ
lap_log = log.filter_by_lap(lap_num)

# サスペンション速度を分析（フィルタ済みのログを受け取る）
result = analyze_suspension_velocity(
    lap_log,
    channel_names=channel_names,
    motion_ratios=motion_ratios,
    velocity_ranges=velocity_ranges,
    smoothing_window=5,  # ノイズ低減用の移動平均
    bin_size=10.0,  # ヒストグラムのビンサイズ（mm/s）
)

print("分析完了！")

ラップ 14 を分析中 - 2:05.056
分析完了！


## 速度ヒストグラム可視化

この4象限プロットは各コーナーの速度分布を示します：
- **青いバー**: バンプ（圧縮）- 正の速度
- **赤いバー**: リバウンド（伸長）- 負の速度
- **背景シェーディング**: 速度域ゾーン

In [6]:
# 速度ヒストグラムをプロット
fig = plot_suspension_velocity_histogram(
    result,
    title=f"サスペンション速度分布 - ラップ {int(selected_lap['num'])} ({format_lap_time(selected_lap['lap_time'])})",
)
show_fig(fig)

## 統計テーブル

### 各コーナーの統計

各コーナーの主要指標：
- **歪度**: 分布の非対称性（0 = 対称、+ = バンプ側が多い、- = リバウンド側が多い）
- **尖度**: 分布の「尾の重さ」（0 = 正規分布、+ = 重い尾）
- **平均/標準偏差**: 平均速度と広がり
- **各域の%**: 各速度域で過ごした時間の割合

In [7]:
# 各コーナーの統計
stats_df = format_suspension_stats_table(result)
stats_df.style.format(
    {
        "Skew": "{:.2f}",
        "Kurtosis": "{:.2f}",
        "Mean (mm/s)": "{:.1f}",
        "Std (mm/s)": "{:.1f}",
        "Zero Bin %": "{:.1f}",
        "Friction %": "{:.1f}",
        "Slow Bump %": "{:.1f}",
        "Slow Rebound %": "{:.1f}",
        "Fast Bump %": "{:.1f}",
        "Fast Rebound %": "{:.1f}",
        "Curb %": "{:.1f}",
    }
)

,Corner,Skew,Kurtosis,Mean (mm/s),Std (mm/s),Zero Bin %,Friction %,Slow Bump %,Slow Rebound %,Fast Bump %,Fast Rebound %,Curb %
0,FL,2.73,42.77,0.0,27.2,25.9,25.9,27.0,32.8,6.8,7.4,0.2
1,FR,0.20,71.69,0.0,33.2,29.0,29.0,27.1,31.7,5.9,5.8,0.5
2,RL,0.82,28.80,0.0,49.3,19.3,19.3,24.2,23.2,15.7,16.8,0.8
3,RR,-0.75,40.25,0.0,53.2,21.4,21.4,23.0,23.7,15.2,15.6,1.1


### バンプ対リバウンドの対称性

各コーナーのバンプ（圧縮）対リバウンド（伸長）を比較：
- **バンプ/合計 %**: 非摩擦時間のうちバンプに費やした割合（50% = 対称）

In [8]:
# バンプ対リバウンドの対称性
symmetry_df = format_symmetry_table(result)
symmetry_df.style.format(
    {
        "Total Bump %": "{:.1f}",
        "Total Rebound %": "{:.1f}",
        "Bump/Total %": "{:.1f}",
        "Slow Bump %": "{:.1f}",
        "Slow Rebound %": "{:.1f}",
        "Fast Bump %": "{:.1f}",
        "Fast Rebound %": "{:.1f}",
    }
)

,Corner,Total Bump %,Total Rebound %,Bump/Total %,Slow Bump %,Slow Rebound %,Fast Bump %,Fast Rebound %
0,FL,33.8,40.2,45.6,27.0,32.8,6.8,7.4
1,FR,33.0,37.5,46.9,27.1,31.7,5.9,5.8
2,RL,39.9,40.0,50.0,24.2,23.2,15.7,16.8
3,RR,38.2,39.3,49.3,23.0,23.7,15.2,15.6


### 左右および前後の比較

ホイールグループを組み合わせてバランスの問題を特定：
- **左右比較**: (FL + RL) 対 (FR + RR) を組み合わせ - 横方向バランスを表示
- **前後比較**: (FL + FR) 対 (RL + RR) を組み合わせ - 縦方向バランスを表示

正の値は左/前がその域で長い時間を過ごしたことを意味し、負の値は右/後を意味します。

In [9]:
# 左右および前後の比較
comparison_df = format_comparison_table(result)
comparison_df.style.format(
    {
        "Zero Bin Diff": "{:+.1f}",
        "Friction Diff": "{:+.1f}",
        "Slow Bump Diff": "{:+.1f}",
        "Slow Rebound Diff": "{:+.1f}",
        "Fast Bump Diff": "{:+.1f}",
        "Fast Rebound Diff": "{:+.1f}",
        "Curb Diff": "{:+.1f}",
    }
)

,Comparison,Zero Bin Diff,Friction Diff,Slow Bump Diff,Slow Rebound Diff,Fast Bump Diff,Fast Rebound Diff,Curb Diff
0,Left vs Right,-2.6,-2.6,+0.5,+0.3,+0.7,+1.4,-0.3
1,Front vs Rear,+7.1,+7.1,+3.4,+8.9,-9.1,-9.7,-0.6
